# C4：插电混动能量管理策略对比 | C4: PHEV Energy Management Strategy Comparison

> **诚实说明 Honesty Note**：本实验调用 **真实 FASTSim 2.1.5 引擎**（Rust 核心）做整车仿真。
> FASTSim 2.x 的混动控制逻辑固化在 Rust 核心中，Python 层无法做逐秒功率分配；
> 因此三种策略通过 **SOC 工作窗口参数**（`min_soc` / `max_soc`）来等效实现——这是一种**参数化近似**，
> 不是逐秒优化的真 ECMS。近似原理与局限见 §1.3。

> **Honesty Note**: This lab uses the **real FASTSim 2.1.5 engine** (Rust core).
> The hybrid control logic is baked into the Rust core and not step-controllable from Python,
> so the three strategies are emulated via **SOC window parameters** — a parametric
> approximation, not true per-second ECMS optimization. See §1.3 for rationale and limits.



## 一、实验原理 | 1. Principle

### 1.1 插电混动（PHEV）的能量管理问题

PHEV 有两套动力源：**电**（电池+电机，高效但存量有限）和 **油**（发动机+油箱，能量密度高但效率低）。
给定一段行程，问题是：**每一秒用多少电、多少油，才能让总成本最低？**

Two energy sources: **electricity** (efficient, limited) and **fuel** (energy-dense, less efficient).
Question: at each second, how much of each minimizes total cost?

### 1.2 三种经典策略 | Three classic strategies

| 策略 Strategy | 思想 Idea | 行为 Behavior |
|---|---|---|
| **CD** 纯电优先 Charge-Depleting | 有电就用电 | 先纯电跑，SOC 耗到下限才启动发动机 |
| **CS** 电量维持 Charge-Sustaining | 把电留着 | 发动机早介入，SOC 维持在低位窄窗口 |
| **ECMS** 等效燃油消耗最小 Equivalent Consumption Minimization | 油电等价 | 每秒最小化等效燃油 $\dot m_{eq} = \dot m_{fuel} + s(t)\cdot P_{batt}/Q_{lhv}$ |

ECMS 的核心是**等效因子** $s(t)$：把 1 kWh 电折算成"相当于烧了多少油"。
$s$ 大 → 电"贵" → 多用油、保电量；$s$ 小 → 电"便宜" → 多用电。
最优的 $s(t)$ 需要对全程做协同优化（庞特里亚金极小值原理）。

The ECMS core is the **equivalence factor** $s(t)$: converting 1 kWh of electricity
into "equivalent fuel". Large $s$ → electricity is "expensive" → save battery; small $s$ → use battery.

### 1.3 本实验的参数化近似 | Parametric approximation in this lab

FASTSim 2.x 的功率分配在 Rust 核心内，Python 只能调 SOC 窗口。
三种策略的等效实现（同一辆 PHEV vnum=12，同一条 UDDS×3 工况）：

| 策略 | `min_soc` | `max_soc` | 等效含义 |
|---|---|---|---|
| A-CD | 0.25 | 0.90（默认） | 满电出发，纯电优先 → 对应 $s$ 很小 |
| B-CS | 0.25 | 0.35 | 出发即近乎亏电，发动机主导 → 对应 $s$ 很大 |
| C-ECMS-like | **0.50** | 0.90 | 发动机在 SOC 50% 即介入，保留电量缓冲 → 对应中等 $s$，"油电兼顾" |

**局限**：真 ECMS 的 $s(t)$ 随工况逐秒变化；本实验的窗口参数是静态的，
只能近似"早介入/晚介入"的趋势，学不到最优 $s(t)$ 轨迹。E4（动态规划）会补上全局最优视角。

**Limitation**: real ECMS adapts $s(t)$ every second; our static window only
approximates the "early/late engine engagement" trend.

### 1.4 评价指标 | Metrics

- **等效油耗** Equivalent fuel consumption：
  $$FC_{eq} = \frac{V_{fuel} + E_{elec}/8.9}{D} \times 100 \quad [L/100km]$$
  其中 $V_{fuel}=E_{fuel,kJ}/34200$（汽油低热值约 34.2 MJ/L），$8.9$ kWh ≈ 1 L 汽油能量当量。
- **终点 SOC** Final SOC：电量维持能力的直接体现。



## 二、实验目的 | 2. Objectives

1. 理解 PHEV 三种能量管理策略（CD / CS / ECMS）的思想与适用场景；
2. 用真实 FASTSim 引擎量化对比三种策略的等效油耗与 SOC 轨迹；
3. 理解"没有免费午餐"：CD 油耗最低但深度放电，CS 保电但油耗高，ECMS 取折中；
4. 为 E4（动态规划全局最优）建立直觉：ECMS 的等效因子思想是 DP 的在线近似。

1. Understand CD / CS / ECMS ideas and use cases;
2. Quantify equivalent fuel & SOC traces with the real FASTSim engine;
3. See the trade-off: CD = lowest fuel but deep discharge, CS = SOC-safe but thirsty, ECMS = balanced;
4. Build intuition for E4 (dynamic programming): ECMS is the online approximation of DP.



In [ ]:
# ===== C4: PHEV Energy Management Strategy Comparison =====
# Kernel: py310-fastsim | Engine: real FASTSim 2.1.5 (Rust core)
import sys, warnings
warnings.filterwarnings('ignore')  # suppress Rust-core divide-by-zero warnings at cycle joints
sys.path.insert(0, '/home/jovyan/labs')

import numpy as np
from fastsim.vehicle import Vehicle, DEFAULT_VEH_DB
from fastsim import cycle
from fastsim.simdrive import SimDrive
from common.vehicle_library import save_result

print('FASTSim engine ready')

# --- Build a long test cycle: UDDS x3 (36 km), long enough to stress the battery ---
c1 = cycle.Cycle.from_file('udds')
t1 = np.array(c1.time_s)
v1 = np.array(c1.mps)
T = np.concatenate([t1, t1 + t1[-1], t1 + 2 * t1[-1]])
V = np.concatenate([v1, v1, v1])
n = len(T)
long_cyc = cycle.Cycle(T, V, np.zeros(n), np.zeros(n), 'udds_x3')
DIST_KM = float(np.trapezoid(V, T)) / 1000.0
print(f'Test cycle: UDDS x3, distance = {DIST_KM:.1f} km')

# --- PHEV under test: vnum=12 from FASTSim vehicle DB ---
veh0 = Vehicle.from_file(DEFAULT_VEH_DB, vnum=12)
print(f"PHEV vnum=12: ess_max_kwh={float(np.array(veh0.ess_max_kwh).mean()):.1f} kWh, "
      f"default SOC window=[{float(np.array(veh0.min_soc).mean()):.2f}, "
      f"{float(np.array(veh0.max_soc).mean()):.2f}]")



## 三、学生任务 | 3. Student Tasks

### 任务 1：运行三种策略（必做）
运行下面代码，对比 A-CD / B-CS / C-ECMS-like 的等效油耗与终点 SOC。

### 任务 2：调参实验（必做）
把 C 策略的 `min_soc` 分别改为 0.35 / 0.50 / 0.65，重跑，记录等效油耗与终点 SOC。
思考：`min_soc` 越高 → 发动机介入越早 → 油耗和 SOC 如何变化？这对应 ECMS 中等效因子 $s$ 变大还是变小？

Vary C's `min_soc` (0.35 / 0.50 / 0.65) and observe: higher `min_soc` → earlier engine
engagement → how do fuel & SOC change? Does this correspond to larger or smaller $s$ in ECMS?

### 任务 3：换工况（选做）
把 UDDS×3 换成 HWFET×2（高速工况），重跑三种策略。高速下电耗占比如何变化？CD 还稳赢吗？

### 🤔 思考题
1. 如果这段行程终点**确定有充电桩**，选哪种策略？如果**行程长度未知**，选哪种？为什么？
2. 真 ECMS 需要整车控制器每 10ms 求解一次优化，本实验的静态窗口近似丢掉了什么信息？



In [ ]:
# --- Strategy definitions: (label, min_soc, max_soc, description) ---
STRATEGIES = [
    ('A-CD',       0.25, 0.90, 'pure electric first, engine only when depleted'),
    ('B-CS',       0.25, 0.35, 'start nearly depleted, engine dominant'),
    ('C-ECMS-like', 0.50, 0.90, 'engine engages at SOC=50%, balanced'),
]

results = {}
for label, min_soc, max_soc in [(s[0], s[1], s[2]) for s in STRATEGIES]:
    veh = Vehicle.from_file(DEFAULT_VEH_DB, vnum=12)  # fresh vehicle each run
    veh.min_soc = min_soc
    veh.max_soc = max_soc
    sd = SimDrive(long_cyc, veh)
    sd.sim_drive()  # <-- real FASTSim Rust-core simulation

    soc = np.array(sd.soc)
    fuel_L = float(np.array(sd.fuel_kj).sum()) / 34200.0      # gasoline LHV ~34.2 MJ/L
    elec_kwh = float(np.array(sd.electric_kwh_per_mi).mean()) * (DIST_KM / 1.60934)
    equiv_L_per_100km = (fuel_L + elec_kwh / 8.9) / DIST_KM * 100.0  # 8.9 kWh ~= 1 L gasoline
    results[label] = {
        'fuel_L': fuel_L,
        'elec_kwh': elec_kwh,
        'equiv_L_per_100km': equiv_L_per_100km,
        'soc_end': float(soc[-1]),
        'soc_min': float(soc.min()),
    }
    r = results[label]
    print(f"{label:10s} | fuel={r['fuel_L']:5.2f} L | elec={r['elec_kwh']:5.2f} kWh | "
          f"equiv={r['equiv_L_per_100km']:5.2f} L/100km | SOC_end={r['soc_end']:.3f}")

# save to the shared data chain (downstream labs / leaderboard can consume it)
save_result('C4_ems_comparison', {'metrics': results, 'cycle': 'udds_x3', 'vnum': 12})
print('\nResults saved to labs/data/results/C4_ems_comparison.json')



In [ ]:
# --- Plot: SOC traces and equivalent fuel comparison (English labels; no CJK font in image) ---
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Re-run SOC traces for plotting (fresh sims, same params)
for label, min_soc, max_soc in [(s[0], s[1], s[2]) for s in STRATEGIES]:
    veh = Vehicle.from_file(DEFAULT_VEH_DB, vnum=12)
    veh.min_soc = min_soc
    veh.max_soc = max_soc
    sd = SimDrive(long_cyc, veh)
    sd.sim_drive()
    axes[0].plot(np.array(sd.soc), label=label)
axes[0].set_xlabel('Time step')
axes[0].set_ylabel('SOC')
axes[0].set_title('SOC trajectories (UDDS x3)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

labels = list(results.keys())
equivs = [results[k]['equiv_L_per_100km'] for k in labels]
colors = ['#2ca02c', '#d62728', '#1f77b4']
axes[1].bar(labels, equivs, color=colors)
axes[1].set_ylabel('L/100km')
axes[1].set_title('Equivalent fuel consumption')
axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()



In [ ]:
# --- Verification asserts (real-simulation gates, per lab standard) ---
rA, rB, rC = results['A-CD'], results['B-CS'], results['C-ECMS-like']

# 1) The three strategies must produce DIFFERENT outcomes
equivs_rounded = {round(r['equiv_L_per_100km'], 2) for r in results.values()}
assert len(equivs_rounded) == 3, f'Strategies not differentiated: {equivs_rounded}'

# 2) ECMS-like preserves SOC best (highest final SOC) -> best charge-sustaining ability
assert rC['soc_end'] > rA['soc_end'] > rB['soc_end'], (
    f"SOC preservation order broken: C={rC['soc_end']:.3f}, A={rA['soc_end']:.3f}, B={rB['soc_end']:.3f}")

# 3) ECMS-like beats CS on fuel (earlier but smarter engine use than always-on CS)
assert rC['equiv_L_per_100km'] < rB['equiv_L_per_100km'], 'ECMS-like should beat CS on fuel'

# 4) ECMS-like stays within 25% of CD fuel -> "near-optimal fuel, best SOC" composite optimum
assert rC['equiv_L_per_100km'] <= rA['equiv_L_per_100km'] * 1.25, (
    f"ECMS-like fuel {rC['equiv_L_per_100km']:.2f} too far from CD {rA['equiv_L_per_100km']:.2f}")

print('✅ All asserts passed:')
print(f'   - 3 strategies differ: {sorted(equivs_rounded)} L/100km')
print(f'   - SOC preservation: C({rC["soc_end"]:.3f}) > A({rA["soc_end"]:.3f}) > B({rB["soc_end"]:.3f})')
print(f'   - ECMS-like: near-CD fuel ({rC["equiv_L_per_100km"]:.2f} vs {rA["equiv_L_per_100km"]:.2f}) + best SOC')



## 四、验证内容 | 4. Verification Checklist

- [ ] 三种策略等效油耗**互不相同**（断言 1 通过）
- [ ] SOC 保持能力排序：C > A > B（断言 2 通过）
- [ ] ECMS-like 油耗优于 CS（断言 3 通过）
- [ ] ECMS-like 油耗在 CD 的 125% 以内（断言 4 通过）
- [ ] 能解释：为什么 CD 油耗最低？（电驱动效率 ~3× 发动机，且电能等效油耗按 8.9 kWh/L 折算）
- [ ] 能解释：为什么 CS 油耗最高？（发动机长期低负荷低效区工作 + 频繁启停）
- [ ] 结果已存入 `labs/data/results/C4_ems_comparison.json`（数据链）

- [ ] 3 strategies differ (assert 1)
- [ ] SOC preservation order C > A > B (assert 2)
- [ ] ECMS-like beats CS on fuel (assert 3)
- [ ] ECMS-like fuel within 125% of CD (assert 4)
- [ ] Can explain why CD has the lowest fuel and CS the highest
- [ ] Results saved to the shared data chain



## 常见报错说明 | Common Errors

| 报错信息 | 中文解释 | 解决方法 |
|---|---|---|
| `ModuleNotFoundError: No module named 'fastsim'` | 内核选错了，FASTSim 只在 `py310-fastsim` 内核可用 | JupyterLab 右上角把内核切换为 **Python 3.10 (FASTSim)** 后 Restart & Run All |
| `RuntimeWarning: divide by zero...` 刷屏 | 拼接工况在接头处 dt=0，Rust 核心的数值警告，无害 | 本实验开头已用 `warnings.filterwarnings('ignore')` 屏蔽 |
| `KeyError: 'fuel_kj'` 或属性不存在 | FASTSim 版本不对（2.x 才有该字段） | 确认镜像 `ghcr.io/itskys/iev-base:jazzy-24.04`，不要用 pip 自装的 fastsim 3.x |
| 三种策略结果**完全一样** | `veh.min_soc/max_soc` 没生效（可能复用了同一个 Vehicle 对象） | 每次策略新建 `Vehicle.from_file(...)`，不要复用对象 |
| `soc_end` 全是 0.9 / 没放电 | 工况太短，电池没耗完 | 用 UDDS×3（36 km）长工况；单 UDDS 只有 12 km，PHEV 纯电就能跑完 |
| 等效油耗为负或极大 | 单位换算错（kJ→L、mi→km） | 检查：`fuel_L = fuel_kj/34200`，`elec_kwh = kwh_per_mi × (km/1.60934)` |
| `FileNotFoundError: labs/data/results` | 结果目录不存在 | `save_result` 会自动建目录；若手动写文件先 `os.makedirs(..., exist_ok=True)` |



## 🏆 挑战任务 | Challenges

1. **扫等效因子**：把 C 策略的 `min_soc` 从 0.30 以 0.05 步长扫到 0.70，画出"等效油耗–终点SOC"帕累托前沿。最优点在哪个 `min_soc`？
   Sweep C's `min_soc` from 0.30 to 0.70, plot the Pareto frontier of (equiv fuel, final SOC).
2. **高速工况**：换 HWFET×2 重跑。高速下风阻主导，CD 的优势是扩大还是缩小？为什么？
3. **真 ECMS 思考**：如果让你在 Python 里实现逐秒 ECMS（已知未来 10 秒车速），你需要 FASTSim 暴露哪些接口？列出来并说明为什么 Rust 核心目前没给。
   (Hint: per-step power split control, engine efficiency map query, predicted power demand.)
4. **链路实验**：用 D3 导出的真实 OCV-SOC 曲线替换本实验 PHEV 的电池参数（经 `BatteryFASTSimAdapter`），观察策略对比结论是否变化。
   Chain lab: replace the battery with D3's real OCV-SOC export and re-run the comparison.

